# Fiscal Data import

In [ ]:
import logging
from utils.source import get_data

logging.basicConfig(level=logging.INFO)
df = get_data()

In [ ]:
from process.process_fiscal import process_fiscal_data
df = process_fiscal_data(df)

In [ ]:
df

### Drop NA values and duplicates

In [ ]:
# Drop only if comune is not set for italian birthplace
mask = (
    df['fiscal_code'].isna() |
    (df['birth_country'] == 'IT') & df['birth_comune_code'].isna()
)
dropped = df[mask]

In [ ]:
dropped

In [ ]:
# Execute drop
df = df[~mask]

In [ ]:
# Drop duplicates, maintain record with more data
df['_score'] = df.notna().sum(axis=1)
df = df.sort_values('_score', ascending=False).drop_duplicates(subset=['fiscal_code']).drop(columns='_score')

## Export

In [ ]:
EXPORT_FILE = "../../process/data/preprocessed_fiscal_data"

In [ ]:
df.to_csv(EXPORT_FILE + ".csv", index=True)
df.to_parquet(EXPORT_FILE + ".parquet")

# Loading data

In [ ]:
import pandas as pd
df = pd.read_parquet(EXPORT_FILE + ".parquet")
df

In [ ]:
from importing.utils.auth import  get_client
from importing.fiscal_data import import_entity

client = get_client()

In [ ]:
import_entity(df, client)

In [ ]:
df[df['fiscal_code'] == 'CVDPTR70A01D612R']